# 7.25 怎樣判斷多花計算值得？


如果 medium 在新題上的正確率比 high 高，應該選哪個？先把需求說清楚：假設我們要處理大量的有限訂單題，希望在平均計算上限內答對最多題。這時 high 這個名稱沒有優先權，真正重要的是留出的驗證題與測試題上的品質與成本。

先固定模型、提示、題目與選字規則（[1.15](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html) 的 greedy 或抽樣），再讓每個檔位回答相同的題目。逐題保存答案、是否正確、是否遵守硬上限、實際工作量與耗時；比較服務時，還要記版本與量測日期。輪數、token 數與秒數是不同的單位，不能直接相加。多候選的方法要把全部候選與挑選的成本都算進去。

下面是假設的四題測試紀錄，模型與規則已經凍結，只用來教怎麼計算報表。`ok` 是依已知的正確答案打的分，1 是對、0 是錯；`work` 暫時用同一個模型執行的輪數代表工作量，沒有宣稱一輪等於固定的秒數。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
logs = {
    "low": {"ok": [1, 1, 0, 0], "work": [1, 1, 1, 1]},
    "medium": {"ok": [1, 1, 1, 1], "work": [3, 3, 3, 3]},
    "high": {"ok": [1, 1, 1, 0], "work": [5, 5, 5, 5]},
}
for name, row in logs.items():
    print(name, "正確率", sum(row["ok"]) / 4, "平均工作", sum(row["work"]) / 4)

oracle_ok, oracle_work = [], []
for i in range(4):
    right = [row["work"][i] for row in logs.values() if row["ok"][i] == 1]
    oracle_ok.append(1 if right else 0)
    oracle_work.append(min(right) if right else logs["low"]["work"][i])
print("事後挑選", "正確率", sum(oracle_ok) / 4, "平均工作", sum(oracle_work) / 4)

這張人工表裡，medium 是 4/4、high 是 3/4，所以在這四題上多算並沒有改善。這說明為什麼要把 medium 這類檔位也固定下來，每次用同一批題重測、當成比較的基準，而不是預設 high 一定最好。四題太少，也不能當成某個模型的穩定排序。實際評估除了平均分數，還要看各類題的表現、失敗的例子，以及足夠樣本下的延遲分布，避免容易題掩蓋難題。

後半段的「事後挑選」先看正確答案，再替每題挑出答對、而且工作最少的檔位：`right` 收集這一題答對的檔位各用了多少工作，`min(right)` 取最少的那個；`if right` 在清單不是空的時候成立。結果前兩題選 low、後兩題選 medium，得到 4/4、平均只用 2 輪，比任何一個固定檔位都好。這叫**離線理想上界**（oracle）：它告訴我們「如果有一個完美的選擇器」最多能做到多好；但部署時沒有正確答案可看，所以它不是可以實際上線的成績。若某題所有檔位都答錯，`right` 是空清單，這題在上界裡也仍然是錯的（程式這時記下 low 的工作量）。

**Auto** 是服務先根據輸入替每題選預算。技術上可行，但多一個選擇器，就多一份成本與犯錯的機會：看似簡單的題可能有隱藏條件，題目的分布改變也會讓舊規則失準。選擇器看不到正確答案，它的成績頂多接近上面的離線理想上界。

要試 Auto，就用驗證資料選規則與門檻，凍結之後才碰測試資料。若 high 只是允許更多工作，也要記它實際用了多少。對缺少必要資訊的題目，先問清楚可能比多算更合理；第 9 章會把「缺資訊」做成明確的判斷，第 19 章再把機率判斷器與 Looped 解題器接起來。

練習：假設只允許平均 3 輪，low、medium、high 哪些符合上限？medium 是本表三個固定檔位中最好的，但不能用這份測試紀錄再回頭挑規則。接下來把 [C.1–C.5](https://birdhackor.github.io/tiny-perceptron-vlm/C.1.html) 當作必修：依序比較可見步驟、候選份數與驗證，理解工作量究竟花在哪裡。
